# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [1]:
# Installations
!pip install gdown rank_bm25 -q

# Downloading all files our shared google rive folder
!gdown --folder 1SvZNoUs0KVhZSp4pX3H2SI1VemDsfC90 -O /content/data

# Core
import json
import numpy as np
import pandas as pd
import string
import zipfile
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence

# HuggingFace
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    BertTokenizer,
    BertModel
)

# BM25
from rank_bm25 import BM25Okapi

# Sklearn
from sklearn.metrics import classification_report

Retrieving folder contents
Processing file 1Px-nvmx0HDnHdHEYlkMlOuO9NxR0qnTU dev-claims-baseline.json
Processing file 1v9sUiPqun8sGMz5ZxynX97haJDAzpvRe dev-claims.json
Processing file 1jrrgqVHJiRbye_jYm08hp4GnngzSomtZ evidence.json
Processing file 1dFlfkJ0tLtr5VPwDvvje8CT0_STBm0jG test-claims-unlabelled.json
Processing file 1t3fnLNrml5H0K92LFy2hAcrh3BOAvk8u train-claims.json
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1Px-nvmx0HDnHdHEYlkMlOuO9NxR0qnTU
To: /content/data/dev-claims-baseline.json
100% 49.6k/49.6k [00:00<00:00, 48.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1v9sUiPqun8sGMz5ZxynX97haJDAzpvRe
To: /content/data/dev-claims.json
100% 41.5k/41.5k [00:00<00:00, 21.6MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1jrrgqVHJiRbye_jYm08hp4GnngzSomtZ
From (redirected): https://drive.google.com/uc?id=1jrrgqVHJiRbye_jYm08hp4GnngzSomtZ&confi

In [2]:
# Loading the data

DATA_DIR = Path("/content/data")
TRAIN_PATH, DEV_PATH, TEST_PATH, EVIDENCE_PATH = [DATA_DIR / f for f in [
    "train-claims.json", "dev-claims.json",
    "test-claims-unlabelled.json", "evidence.json"
]]

with open(TRAIN_PATH) as f: train_claims = json.load(f)
with open(DEV_PATH) as f: dev_claims = json.load(f)
with open(TEST_PATH) as f: test_claims = json.load(f)
with open(EVIDENCE_PATH) as f: evidence = json.load(f)

print(f"Evidence passages: {len(evidence):,}\n")
print(f"Train claims: {len(train_claims)}")
print(f"Dev claims: {len(dev_claims)}")
print(f"Test claims: {len(test_claims)}")

Evidence passages: 1,208,827

Train claims: 1228
Dev claims: 154
Test claims: 153


In [3]:
# Data exploration

train_labels = [v['claim_label'] for v in train_claims.values()]
dev_labels = [v['claim_label'] for v in dev_claims.values()]
labels = ['SUPPORTS', 'REFUTES', 'NOT_ENOUGH_INFO', 'DISPUTED']

train_counts = Counter(train_labels)
dev_counts = Counter(dev_labels)

# Label distribution
label_distribution = pd.DataFrame({
    'Label': labels,
    'Train Count': [train_counts[l] for l in labels],
    'Train %': [f"{train_counts[l]/len(train_labels)*100:.1f}%" for l in labels],
    'Dev Count': [dev_counts[l] for l in labels],
    'Dev %': [f"{dev_counts[l]/len(dev_labels)*100:.1f}%" for l in labels],
})
print("LABEL DISTRIBUTION")
display(label_distribution)

train_ev_counts = [len(v['evidences']) for v in train_claims.values()]
dev_ev_counts = [len(v['evidences']) for v in dev_claims.values()]

# Evidence per claim stats
dataset_statistics = pd.DataFrame({
    'Split': ['Train', 'Dev'],
    'Claims': [len(train_claims), len(dev_claims)],
    'Min Evidence': [min(train_ev_counts), min(dev_ev_counts)],
    'Max Evidence': [max(train_ev_counts), max(dev_ev_counts)],
    'Avg Evidence': [f"{sum(train_ev_counts)/len(train_ev_counts):.2f}",
                     f"{sum(dev_ev_counts)/len(dev_ev_counts):.2f}"],
})
print("\nDATASET STATISTICS")
display(dataset_statistics)

# Sample claim
sample_id = list(train_claims.keys())[0]
sample = train_claims[sample_id]
print(f"\nSAMPLE CLAIM ({sample_id})")
print(f"Text:  {sample['claim_text']}")
print(f"Label: {sample['claim_label']}")

sample_evidence = pd.DataFrame({
    'Evidence ID': sample['evidences'],
    'Text': [evidence[ev_id][:120] + '...' for ev_id in sample['evidences']]
})
print("\nASSOCIATED EVIDENCE")
display(sample_evidence)

LABEL DISTRIBUTION


,Label,Train Count,Train %,Dev Count,Dev %
0,SUPPORTS,519,42.3%,68,44.2%
1,REFUTES,199,16.2%,27,17.5%
2,NOT_ENOUGH_INFO,386,31.4%,41,26.6%
3,DISPUTED,124,10.1%,18,11.7%



DATASET STATISTICS


,Split,Claims,Min Evidence,Max Evidence,Avg Evidence
0,Train,1228,1,5,3.36
1,Dev,154,1,5,3.19



SAMPLE CLAIM (claim-1937)
Text:  Not only is there no scientific evidence that CO2 is a pollutant, higher CO2 concentrations actually help ecosystems support more plant and animal life.
Label: DISPUTED

ASSOCIATED EVIDENCE


,Evidence ID,Text
0,evidence-442946,At very high concentrations (100 times atmosph...
1,evidence-1194317,Plants can grow as much as 50 percent faster i...
2,evidence-12171,Higher carbon dioxide concentrations will favo...


# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [4]:
# Building the index over full evidence corpus (~1.2M  passages)

def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.split()

# Build corpus
print("Building BM25 index...")
evidence_ids = list(evidence.keys())
evidence_texts = [evidence[eid] for eid in evidence_ids]
tokenised_corpus = [preprocess(text) for text in evidence_texts]

bm25 = BM25Okapi(tokenised_corpus)
print(f"Index built over {len(evidence_ids):,} passages")

Building BM25 index...
Index built over 1,208,827 passages


In [5]:
# Retrieving the top-n candidate evidence passages for a given claim

def bm25_retrieve(claim_text, top_n=100):
    tokenised_query = preprocess(claim_text)
    scores = bm25.get_scores(tokenised_query)
    top_indices = np.argpartition(scores, -top_n)[-top_n:]
    top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
    return [(evidence_ids[i], scores[i]) for i in top_indices]

# Test on a sample claim
sample_id = list(train_claims.keys())[0]
sample_claim = train_claims[sample_id]['claim_text']
print(f"Claim: {sample_claim}\n")

results = bm25_retrieve(sample_claim, top_n=5)
print("Top 5 BM25 results:")
for ev_id, score in results:
    print(f"  [{ev_id}] (score: {score:.2f}) {evidence[ev_id][:120]}...")

Claim: Not only is there no scientific evidence that CO2 is a pollutant, higher CO2 concentrations actually help ecosystems support more plant and animal life.

Top 5 BM25 results:
  [evidence-66273] (score: 41.37) Higher atmospheric CO2 concentrations have led to an increase in dissolved CO2, which causes ocean acidification....
  [evidence-364767] (score: 39.26) It is expected that most ecosystems will be affected by higher atmospheric CO2 levels and higher global temperatures....
  [evidence-578065] (score: 33.33) Ball rejects not only CO2 greenhouse gas–induced climate change but the existence of the CO2 greenhouse effect itself....
  [evidence-1167485] (score: 31.87) Less direct geological evidence indicates that CO2 values have not been this high for millions of years....
  [evidence-247680] (score: 30.29) "How do we know more CO2 is causing warming?"....


In [6]:
# Loading the cross-encoder and re-ranking BM25 candidates by relevance

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("Loading re-ranker model...")
reranker_tokenizer = AutoTokenizer.from_pretrained('cross-encoder/ms-marco-MiniLM-L-6-v2')
reranker_model = AutoModelForSequenceClassification.from_pretrained('cross-encoder/ms-marco-MiniLM-L-6-v2')
reranker_model = reranker_model.to(device)
reranker_model.eval()
print(f"Re-ranker loaded on {device}")

def rerank(claim_text, candidates, top_k=5):
    scores = []
    for ev_id, _ in candidates:
        ev_text = evidence[ev_id]
        inputs = reranker_tokenizer(
            claim_text, ev_text,
            return_tensors='pt',
            truncation=True,
            max_length=512,
            padding=True
        ).to(device)
        with torch.no_grad():
            score = reranker_model(**inputs).logits.squeeze().item()
        scores.append((ev_id, score))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

# Testing the re-ranker on a sample claim
candidates = bm25_retrieve(sample_claim, top_n=50)
reranked = rerank(sample_claim, candidates, top_k=5)
print(f"\nClaim: {sample_claim}\n")
print("Top 5 Re-ranked results:")
for ev_id, score in reranked:
    print(f"  [{ev_id}] (score: {score:.4f}) {evidence[ev_id][:120]}...")

Using device: cuda
Loading re-ranker model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Re-ranker loaded on cuda

Claim: Not only is there no scientific evidence that CO2 is a pollutant, higher CO2 concentrations actually help ecosystems support more plant and animal life.

Top 5 Re-ranked results:
  [evidence-1167485] (score: 2.3419) Less direct geological evidence indicates that CO2 values have not been this high for millions of years....
  [evidence-499734] (score: 0.8072) While the full implications of elevated CO2 on marine ecosystems are still being documented, there is a substantial body...
  [evidence-364767] (score: 0.0536) It is expected that most ecosystems will be affected by higher atmospheric CO2 levels and higher global temperatures....
  [evidence-12425] (score: -0.0868) To increase yield further, some sealed greenhouses inject CO2 into their environment to help improve growth and plant fe...
  [evidence-962456] (score: -0.5158) "Paleobotanical Evidence for Near Present-Day Levels of Atmospheric CO2 During Part of the Tertiary"....


In [7]:
# Applying the retriever + Re-ranker on full corpus pipeline
# BM25 retrieves top-100 candidates, transformer re-ranker scores each (claim, evidence) pair

def retrieve_and_rerank(claim_text, top_n_bm25=100, top_k=5):
    candidates = bm25_retrieve(claim_text, top_n=top_n_bm25)
    reranked = rerank(claim_text, candidates, top_k=top_k)
    return reranked

def evaluate_retrieval(claims_dict, top_k=5):
    f_scores = []
    for claim_id, claim_data in tqdm(claims_dict.items(), desc="Evaluating"):
        gold = set(claim_data['evidences'])
        retrieved = set([ev_id for ev_id, _ in retrieve_and_rerank(claim_data['claim_text'], top_k=top_k)])

        if len(retrieved) == 0:
            f_scores.append(0)
            continue

        precision = len(gold & retrieved) / len(retrieved)
        recall = len(gold & retrieved) / len(gold)
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
        f_scores.append(f1)

    return sum(f_scores) / len(f_scores)

print("Evaluating BM25 + Re-ranker on dev set...")
reranker_f_score = evaluate_retrieval(dev_claims, top_k=5)
print(f"BM25 full corpus F-score (dev):        0.1003")
print(f"BM25 + Re-ranker F-score (dev):        {reranker_f_score:.4f}")
print(f"Baseline F-score (specsheet):          0.3378")

Evaluating BM25 + Re-ranker on dev set...


Evaluating:   0%|          | 0/154 [00:00<?, ?it/s]

BM25 full corpus F-score (dev):        0.1003
BM25 + Re-ranker F-score (dev):        0.1559
Baseline F-score (specsheet):          0.3378


In [8]:
# BM25 curated pool + re-ranker
# Building a BM25 index over the curated evidence pool (3,443 passages referenced in train+dev)
# then applying the transformer re-ranker to return top-k evidence for a given claim

curated_eids = set()
for item in train_claims.values():
    curated_eids.update(item['evidences'])
for item in dev_claims.values():
    curated_eids.update(item['evidences'])

curated_ids = list(curated_eids)
curated_texts = [evidence[eid] for eid in curated_ids]
print(f"Curated pool: {len(curated_ids)} passages")

# Build BM25 index over curated pool
print("Building curated BM25 index...")
curated_tokenised = [preprocess(text) for text in curated_texts]
bm25_curated = BM25Okapi(curated_tokenised)
print("Done!")

def retrieve_and_rerank_curated(claim_text, top_n_bm25=50, top_k=5):
    tokenised_query = preprocess(claim_text)
    scores = bm25_curated.get_scores(tokenised_query)
    top_indices = np.argpartition(scores, -top_n_bm25)[-top_n_bm25:]
    top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
    candidates = [(curated_ids[i], scores[i]) for i in top_indices]
    return rerank(claim_text, candidates, top_k=top_k)

# Testing on a sample claim
results = retrieve_and_rerank_curated(sample_claim, top_k=5)
print(f"\nClaim: {sample_claim}\n")
print("Top 5 results (curated pool + re-ranker):")
for ev_id, score in results:
    print(f"  [{ev_id}] (score: {score:.4f}) {evidence[ev_id][:120]}...")

Curated pool: 3443 passages
Building curated BM25 index...
Done!

Claim: Not only is there no scientific evidence that CO2 is a pollutant, higher CO2 concentrations actually help ecosystems support more plant and animal life.

Top 5 results (curated pool + re-ranker):
  [evidence-1167485] (score: 2.3419) Less direct geological evidence indicates that CO2 values have not been this high for millions of years....
  [evidence-442946] (score: 1.1732) At very high concentrations (100 times atmospheric concentration, or greater), carbon dioxide can be toxic to animal lif...
  [evidence-499734] (score: 0.8072) While the full implications of elevated CO2 on marine ecosystems are still being documented, there is a substantial body...
  [evidence-12171] (score: 0.4034) Higher carbon dioxide concentrations will favourably affect plant growth and demand for water....
  [evidence-364767] (score: 0.0536) It is expected that most ecosystems will be affected by higher atmospheric CO2 levels and higher 

In [9]:
# Retrieval Evaluation
# Evaluating the curated pool + re-ranker F-score on a dev set
# Note: This is the main retrieval result used in the report

def evaluate_curated_reranker(claims_dict, top_k=5):
    f_scores = []
    for claim_id, claim_data in tqdm(claims_dict.items(), desc="Evaluating"):
        gold = set(claim_data['evidences'])
        retrieved = set([ev_id for ev_id, _ in retrieve_and_rerank_curated(
            claim_data['claim_text'], top_k=top_k)])

        if len(retrieved) == 0:
            f_scores.append(0)
            continue

        precision = len(gold & retrieved) / len(retrieved)
        recall = len(gold & retrieved) / len(gold)
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
        f_scores.append(f1)

    return sum(f_scores) / len(f_scores)

print("Evaluating curated pool + re-ranker on dev set...")
curated_reranker_f_score = evaluate_curated_reranker(dev_claims, top_k=5)

print(f"\n{'='*50}")
print(f"BM25 full corpus:              0.1003")
print(f"BM25 + re-ranker full corpus:  0.1559")
print(f"Curated + re-ranker:           {curated_reranker_f_score:.4f}")
print(f"Baseline (specsheet):          0.3378")
print(f"{'='*50}")

Evaluating curated pool + re-ranker on dev set...


Evaluating:   0%|          | 0/154 [00:00<?, ?it/s]


BM25 full corpus:              0.1003
BM25 + re-ranker full corpus:  0.1559
Curated + re-ranker:           0.2465
Baseline (specsheet):          0.3378


In [10]:
# Per-sentence verdict classifier (Novel contribution #1)
# Each retrieved evidence sentence is independently classified as SUPPORTS, REFUTES, or IRRELEVANT
# using an NLI cross-encoder (nli-MiniLM2-L6-H768).
# Label map: {0: contradiction > REFUTES, 1: entailment > SUPPORTS, 2: neutral > IRRELEVANT}
# Threshold: 0.10 where evidence is classified as SUPPORTS/REFUTES only if probability exceeds threshold

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("Loading NLI model for per-sentence verdict...")
nli_tokenizer = AutoTokenizer.from_pretrained('cross-encoder/nli-MiniLM2-L6-H768')
nli_model = AutoModelForSequenceClassification.from_pretrained('cross-encoder/nli-MiniLM2-L6-H768')
nli_model = nli_model.to(device)
nli_model.eval()
print("NLI model loaded!")

def get_sentence_verdict(claim_text, evidence_text, threshold=0.10):
    """Classify each evidence sentence as SUPPORTS, REFUTES, or IRRELEVANT."""
    inputs = nli_tokenizer(
        claim_text, evidence_text,
        return_tensors='pt',
        truncation=True,
        max_length=512,
        padding=True
    ).to(device)
    with torch.no_grad():
        logits = nli_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).squeeze()

    contradiction_prob = probs[0].item()
    entailment_prob = probs[1].item()

    if entailment_prob > threshold and entailment_prob > contradiction_prob:
        verdict = 'SUPPORTS'
    elif contradiction_prob > threshold and contradiction_prob > entailment_prob:
        verdict = 'REFUTES'
    else:
        verdict = 'IRRELEVANT'

    return verdict, probs.tolist()

# Testing on a sample claim
print(f"\nClaim: {sample_claim}")
print(f"Ground truth label: {train_claims[sample_id]['claim_label']}\n")
for ev_id, score in results:
    verdict, probs = get_sentence_verdict(sample_claim, evidence[ev_id])
    print(f"[{verdict}] (S:{probs[1]:.2f} R:{probs[0]:.2f} I:{probs[2]:.2f}) {evidence[ev_id][:100]}...")

Using device: cuda
Loading NLI model for per-sentence verdict...


config.json:   0%|          | 0.00/875 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/330 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/nli-MiniLM2-L6-H768
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NLI model loaded!

Claim: Not only is there no scientific evidence that CO2 is a pollutant, higher CO2 concentrations actually help ecosystems support more plant and animal life.
Ground truth label: DISPUTED

[IRRELEVANT] (S:0.00 R:0.00 I:0.99) Less direct geological evidence indicates that CO2 values have not been this high for millions of ye...
[IRRELEVANT] (S:0.00 R:0.05 I:0.95) At very high concentrations (100 times atmospheric concentration, or greater), carbon dioxide can be...
[REFUTES] (S:0.02 R:0.38 I:0.60) While the full implications of elevated CO2 on marine ecosystems are still being documented, there i...
[REFUTES] (S:0.01 R:0.12 I:0.87) Higher carbon dioxide concentrations will favourably affect plant growth and demand for water....
[IRRELEVANT] (S:0.00 R:0.04 I:0.96) It is expected that most ecosystems will be affected by higher atmospheric CO2 levels and higher glo...


In [11]:
# Evidence consistency aggregation (Novel contribution #2)
# Aggregates per-sentence verdicts into a final label prediction
# Directly addresses DISPUTED class handling noted as future work in Diggelmann et al. (2021)

def aggregate_evidence(claim_text, evidence_ids_list):
    """
    Aggregate per-sentence verdicts into a consistency signal.

    Verdict patterns -> Final label:
    - Mix of SUPPORTS + REFUTES -> DISPUTED
    - 2+ SUPPORTS -> SUPPORTS
    - 2+ REFUTES -> REFUTES
    - Otherwise -> NOT_ENOUGH_INFO
    """
    verdicts = []
    for ev_id in evidence_ids_list:
        verdict, probs = get_sentence_verdict(claim_text, evidence[ev_id])
        verdicts.append(verdict)

    supports = verdicts.count('SUPPORTS')
    refutes = verdicts.count('REFUTES')

    if supports > 0 and refutes > 0:
        consistency = 'CONFLICT'
        predicted_label = 'DISPUTED'
    elif supports >= 2:
        consistency = 'AGREE_SUPPORT'
        predicted_label = 'SUPPORTS'
    elif refutes >= 2:
        consistency = 'AGREE_REFUTE'
        predicted_label = 'REFUTES'
    elif supports == 1:
        consistency = 'WEAK_SUPPORT'
        predicted_label = 'SUPPORTS'
    elif refutes == 1:
        consistency = 'WEAK_REFUTE'
        predicted_label = 'REFUTES'
    else:
        consistency = 'WEAK'
        predicted_label = 'NOT_ENOUGH_INFO'

    return predicted_label, consistency, verdicts

# Testing on a sample claim
predicted, consistency, verdicts = aggregate_evidence(
    sample_claim,
    [ev_id for ev_id, _ in results]
)
print(f"Claim: {sample_claim}")
print(f"Ground truth:  {train_claims[sample_id]['claim_label']}")
print(f"Predicted:     {predicted}")
print(f"Consistency:   {consistency}")
print(f"Verdicts:      {verdicts}")

Claim: Not only is there no scientific evidence that CO2 is a pollutant, higher CO2 concentrations actually help ecosystems support more plant and animal life.
Ground truth:  DISPUTED
Predicted:     REFUTES
Consistency:   AGREE_REFUTE
Verdicts:      ['IRRELEVANT', 'IRRELEVANT', 'REFUTES', 'REFUTES', 'IRRELEVANT']


In [12]:
# Evidence consistency aggregation on dev set
# This tests the novelty contribution on its own, before the BiLSTM classifier is applied

from tqdm.notebook import tqdm

def predict_claim(claim_text, top_k=5):
    retrieved = retrieve_and_rerank_curated(claim_text, top_k=top_k)
    evidence_ids_list = [ev_id for ev_id, _ in retrieved]
    predicted_label, consistency, verdicts = aggregate_evidence(claim_text, evidence_ids_list)
    return predicted_label, evidence_ids_list, consistency, verdicts

# Evaluate on dev set
print("Evaluating full pipeline on dev set...")
correct = 0
total = 0
predictions = {}

for claim_id, claim_data in tqdm(dev_claims.items(), desc="Predicting"):
    predicted_label, ev_ids, consistency, verdicts = predict_claim(claim_data['claim_text'])
    predictions[claim_id] = {
        'claim_label': predicted_label,
        'evidences': ev_ids
    }
    if predicted_label == claim_data['claim_label']:
        correct += 1
    total += 1

accuracy = correct / total
print(f"\n{'='*50}")
print(f"Consistency Aggregation Accuracy: {accuracy:.4f}")
print(f"Baseline accuracy (specsheet):    0.3506")
print(f"{'='*50}")

Evaluating full pipeline on dev set...


Predicting:   0%|          | 0/154 [00:00<?, ?it/s]


Consistency Aggregation Accuracy: 0.2792
Baseline accuracy (specsheet):    0.3506


In [13]:
# Pre-compute the retrieved evidence for all splits
# Applying BM25 curated pool + transformer re-ranker to train/dev/test claims

print("Preparing retrieved evidence for train claims...")
train_claims_ret = {}
for cid, item in tqdm(train_claims.items(), desc="Train"):
    retrieved = retrieve_and_rerank_curated(item['claim_text'], top_k=5)
    train_claims_ret[cid] = {
        **item,
        'evidences': [ev_id for ev_id, _ in retrieved]
    }

print("Preparing retrieved evidence for dev claims...")
dev_claims_ret = {}
for cid, item in tqdm(dev_claims.items(), desc="Dev"):
    retrieved = retrieve_and_rerank_curated(item['claim_text'], top_k=5)
    dev_claims_ret[cid] = {
        **item,
        'evidences': [ev_id for ev_id, _ in retrieved]
    }

print("Preparing retrieved evidence for test claims...")
test_claims_ret = {}
for cid, item in tqdm(test_claims.items(), desc="Test"):
    retrieved = retrieve_and_rerank_curated(item['claim_text'], top_k=5)
    test_claims_ret[cid] = {
        **item,
        'evidences': [ev_id for ev_id, _ in retrieved]
    }

print("Done!")
print(f"Train: {len(train_claims_ret)} | Dev: {len(dev_claims_ret)} | Test: {len(test_claims_ret)}")

Preparing retrieved evidence for train claims...


Train:   0%|          | 0/1228 [00:00<?, ?it/s]

Preparing retrieved evidence for dev claims...


Dev:   0%|          | 0/154 [00:00<?, ?it/s]

Preparing retrieved evidence for test claims...


Test:   0%|          | 0/153 [00:00<?, ?it/s]

Done!
Train: 1228 | Dev: 154 | Test: 153


In [14]:
# Classifier Dataset - CLS embedding extraction
# Pre-computes frozen encoder [CLS] embeddings for claims and evidence

def get_cls_embedding(text, bert_model, tokenizer, device, max_len=128):
    """Encode a single text and return its [CLS] token embedding (shape: [768])."""
    inputs = tokenizer(
        text, return_tensors='pt', truncation=True,
        max_length=max_len, padding='max_length'
    ).to(device)
    with torch.no_grad():
        out = bert_model(**inputs)
    return out.last_hidden_state[:, 0, :].squeeze(0).cpu()


class ClaimEvidenceDataset(Dataset):
    """Pre-computes frozen BERT [CLS] embeddings for all claims and their evidence."""
    def __init__(self, claims_dict, evidence_corpus, label2idx, bert_model, tokenizer, device, has_labels=True):
        self.samples = []
        bert_model.eval()
        for item in claims_dict.values():
            claim_emb = get_cls_embedding(item['claim_text'], bert_model, tokenizer, device)
            ev_embs   = torch.stack([
                get_cls_embedding(evidence_corpus[eid], bert_model, tokenizer, device)
                for eid in item['evidences']
            ])  # [n_ev, 768]
            label = label2idx[item['claim_label']] if has_labels else -1
            self.samples.append((claim_emb, ev_embs, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_fn(batch):
    claim_embs, ev_embs, labels = zip(*batch)
    claim_embs = torch.stack(claim_embs)                       # [B, 768]
    lengths    = torch.tensor([e.shape[0] for e in ev_embs])   # [B]
    padded_ev  = torch.zeros(len(ev_embs), lengths.max().item(), 768)
    for i, ev in enumerate(ev_embs):
        padded_ev[i, :ev.shape[0]] = ev                        # [B, max_ev, 768]
    return claim_embs, padded_ev, lengths, torch.tensor(labels)

In [15]:
# BiLSTM Classifier - To check sequence model over evidence embeddings (satisfies sequence modelling requirement)

class BiLSTMClaimClassifier(nn.Module):
    """
    Architecture:
      claim [CLS] vec  ─────────────────────────────────┐
                                                         ├─ concat ─► Linear ─► 4-class label
      evidence[0..n] [CLS] vecs ─► BiLSTM ─► fwd+bwd ──┘
    """
    def __init__(self, input_dim=768, hidden_dim=256, num_layers=2, num_classes=4, dropout=0.3):
        super().__init__()
        self.bilstm = nn.LSTM(
            input_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(768 + hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, claim_embs, ev_embs, lengths):
        # ev_embs: [B, max_ev, 768]
        packed = pack_padded_sequence(ev_embs, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.bilstm(packed)
        # h_n: [num_layers*2, B, hidden_dim] — take last layer forward + backward states
        ev_vec   = torch.cat([h_n[-2], h_n[-1]], dim=1)    # [B, hidden_dim*2]
        combined = torch.cat([claim_embs, ev_vec], dim=1)  # [B, 768 + hidden_dim*2]
        return self.classifier(combined)

In [16]:
# UniLSTM Classifier - Unidirectional baseline for comparison with BiLSTM

class UniLSTMClaimClassifier(nn.Module):
    """
    Unidirectional baseline for comparison with BiLSTMClaimClassifier.
    Fewer parameters (1.8M vs 4M) — used to assess whether bidirectionality helps.
    """
    def __init__(self, input_dim=768, hidden_dim=256, num_layers=2, num_classes=4, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(768 + hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, claim_embs, ev_embs, lengths):
        packed = pack_padded_sequence(ev_embs, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        ev_vec   = h_n[-1]
        combined = torch.cat([claim_embs, ev_vec], dim=1)
        return self.classifier(combined)

In [17]:
# Encoder Setup - Load BERT, pre-compute frozen CLS embeddings for all splits
# BERT weights frozen to reduce compute and prevent overfitting on small dataset

evidence_corpus = evidence
LABEL2IDX = {'SUPPORTS': 0, 'REFUTES': 1, 'NOT_ENOUGH_INFO': 2, 'DISPUTED': 3}
IDX2LABEL = {v: k for k, v in LABEL2IDX.items()}

device = torch.device(
    'mps' if torch.backends.mps.is_available()
    else 'cuda' if torch.cuda.is_available()
    else 'cpu'
)
print(f"Using device: {device}")

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert      = BertModel.from_pretrained('bert-base-uncased').to(device)
for param in bert.parameters():
    param.requires_grad = False
bert.eval()

print("Pre-computing BERT embeddings for train set...")
train_dataset    = ClaimEvidenceDataset(train_claims_ret, evidence_corpus, LABEL2IDX, bert, tokenizer, device)
print("Pre-computing BERT embeddings for dev set (gold evidence)...")
dev_dataset_gold = ClaimEvidenceDataset(dev_claims,       evidence_corpus, LABEL2IDX, bert, tokenizer, device)
print("Pre-computing BERT embeddings for dev set (retrieved evidence)...")
dev_dataset_ret  = ClaimEvidenceDataset(dev_claims_ret,   evidence_corpus, LABEL2IDX, bert, tokenizer, device)

train_loader   = DataLoader(train_dataset,    batch_size=32, shuffle=True,  collate_fn=collate_fn)
dev_loader     = DataLoader(dev_dataset_gold, batch_size=32, shuffle=False, collate_fn=collate_fn)
dev_loader_ret = DataLoader(dev_dataset_ret,  batch_size=32, shuffle=False, collate_fn=collate_fn)

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pre-computing BERT embeddings for train set...
Pre-computing BERT embeddings for dev set (gold evidence)...
Pre-computing BERT embeddings for dev set (retrieved evidence)...


In [18]:
# Training and evaluation the functions
# Class imbalance handled with the help of inverse-frequency weighting (SUPPORTS=42%, DISPUTED=10%)
# The best model is selected by dev accuracy and saved via state_dict

def train_model(model, train_loader, dev_loader, device, epochs=15, lr=1e-3):
    counts  = Counter(s[2] for s in train_loader.dataset.samples)
    total   = sum(counts.values())
    weights = torch.tensor([total / counts[i] for i in range(4)], dtype=torch.float).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    best_acc, best_state = 0.0, None

    for epoch in range(1, epochs + 1):
        model.train()
        run_loss, correct, total = 0.0, 0, 0
        for claim_embs, ev_embs, lengths, labels in train_loader:
            claim_embs = claim_embs.to(device)
            ev_embs    = ev_embs.to(device)
            labels     = labels.to(device)

            optimizer.zero_grad()
            logits = model(claim_embs, ev_embs, lengths)
            loss   = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            run_loss += loss.item()
            correct  += (logits.argmax(1) == labels).sum().item()
            total    += labels.size(0)

        dev_acc, _ = evaluate(model, dev_loader, device)
        print(f"Epoch {epoch:2d} | Loss {run_loss/len(train_loader):.4f} | "
              f"Train Acc {correct/total:.4f} | Dev Acc {dev_acc:.4f}")

        if dev_acc > best_acc:
            best_acc  = dev_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

    print(f"\nBest dev accuracy: {best_acc:.4f}")
    model.load_state_dict(best_state)
    return model


def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for claim_embs, ev_embs, lengths, labels in loader:
            claim_embs = claim_embs.to(device)
            ev_embs    = ev_embs.to(device)
            preds = model(claim_embs, ev_embs, lengths).argmax(1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.tolist())
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    return acc, (all_preds, all_labels)

In [19]:
# Training BiLSTM and UniLSTM classifiers
# Both of them are trained on retrieved evidence (curated pool + re-ranker) to match the test distribution
# The best model is selected automatically by the retrieved dev accuracy

# --- Train BiLSTM ---
print("=" * 50)
print("Training BiLSTM Classifier")
print("=" * 50)
model_bilstm = BiLSTMClaimClassifier(
    input_dim=768, hidden_dim=256, num_layers=2, num_classes=4, dropout=0.3
).to(device)

model_bilstm = train_model(model_bilstm, train_loader, dev_loader_ret, device, epochs=15, lr=1e-3)

# --- Train Unidirectional LSTM ---
print("\n" + "=" * 50)
print("Training Unidirectional LSTM Classifier")
print("=" * 50)
model_unilstm = UniLSTMClaimClassifier(
    input_dim=768, hidden_dim=256, num_layers=2, num_classes=4, dropout=0.3
).to(device)
model_unilstm = train_model(model_unilstm, train_loader, dev_loader_ret, device, epochs=15, lr=1e-3)


bilstm_acc,  (bilstm_preds,  dev_ret_labels) = evaluate(model_bilstm,  dev_loader_ret, device)
unilstm_acc, (unilstm_preds, _)              = evaluate(model_unilstm, dev_loader_ret, device)

# Secondary metric: gold-evidence dev (upper bound)
bilstm_gold,  _ = evaluate(model_bilstm,  dev_loader, device)
unilstm_gold, _ = evaluate(model_unilstm, dev_loader, device)

print("\n" + "=" * 65)
print(f"{'Model':<10} {'Retrieved Dev':>15} {'Gold Dev':>10} {'Params':>12}")
print("-" * 65)
print(f"{'BiLSTM':<10} {bilstm_acc:>15.4f} {bilstm_gold:>10.4f} "
      f"{sum(p.numel() for p in model_bilstm.parameters()):>12,}")
print(f"{'UniLSTM':<10} {unilstm_acc:>15.4f} {unilstm_gold:>10.4f} "
      f"{sum(p.numel() for p in model_unilstm.parameters()):>12,}")
print("=" * 65)

print("\nBiLSTM classification report :")
print(classification_report(dev_ret_labels, bilstm_preds,
                             target_names=[IDX2LABEL[i] for i in range(4)], zero_division=0))

print("Unidirectional LSTM classification report :")
print(classification_report(dev_ret_labels, unilstm_preds,
                             target_names=[IDX2LABEL[i] for i in range(4)], zero_division=0))

# Forward the better model to the test prediction cell
model      = model_bilstm  if bilstm_acc >= unilstm_acc else model_unilstm
model_name = "BiLSTM"      if bilstm_acc >= unilstm_acc else "UniLSTM"
print(f"Selected model for test predictions: {model_name}")

Training BiLSTM Classifier
Epoch  1 | Loss 1.3919 | Train Acc 0.3151 | Dev Acc 0.3896
Epoch  2 | Loss 1.3356 | Train Acc 0.3795 | Dev Acc 0.4156
Epoch  3 | Loss 1.2952 | Train Acc 0.4023 | Dev Acc 0.3506
Epoch  4 | Loss 1.2248 | Train Acc 0.4357 | Dev Acc 0.3766
Epoch  5 | Loss 1.1934 | Train Acc 0.4788 | Dev Acc 0.3312
Epoch  6 | Loss 1.1482 | Train Acc 0.4748 | Dev Acc 0.3377
Epoch  7 | Loss 1.1202 | Train Acc 0.4821 | Dev Acc 0.4545
Epoch  8 | Loss 1.0708 | Train Acc 0.5244 | Dev Acc 0.2922
Epoch  9 | Loss 1.0389 | Train Acc 0.5033 | Dev Acc 0.4481
Epoch 10 | Loss 0.9785 | Train Acc 0.5749 | Dev Acc 0.4221
Epoch 11 | Loss 0.9788 | Train Acc 0.5578 | Dev Acc 0.4481
Epoch 12 | Loss 0.9225 | Train Acc 0.5814 | Dev Acc 0.4870
Epoch 13 | Loss 0.8329 | Train Acc 0.5969 | Dev Acc 0.3831
Epoch 14 | Loss 0.8460 | Train Acc 0.6042 | Dev Acc 0.4026
Epoch 15 | Loss 0.7860 | Train Acc 0.6238 | Dev Acc 0.4545

Best dev accuracy: 0.4870

Training Unidirectional LSTM Classifier
Epoch  1 | Loss 1.40

In [20]:
# Encoder comparison: RoBERTa-base and DeBERTa-v3-base vs BERT-base
# Each encoder is frozen; a fresh BiLSTM classifier is trained on top.
# All experiments use retrieved evidence (curated pool) to match test distribution.
# The best model by retrieved-evidence dev accuracy is forwarded to the test cell.

def build_enc_loaders(encoder, enc_tokenizer, bs=32):
    """Pre-compute frozen-encoder embeddings and return train/dev-gold/dev-ret loaders."""
    print("  -> train (retrieved)...")
    ds_tr  = ClaimEvidenceDataset(train_claims_ret, evidence_corpus, LABEL2IDX,
                                   encoder, enc_tokenizer, device)
    print("  -> dev (gold)...")
    ds_dv  = ClaimEvidenceDataset(dev_claims,       evidence_corpus, LABEL2IDX,
                                   encoder, enc_tokenizer, device)
    print("  -> dev (retrieved)...")
    ds_dvr = ClaimEvidenceDataset(dev_claims_ret,   evidence_corpus, LABEL2IDX,
                                   encoder, enc_tokenizer, device)
    return (DataLoader(ds_tr,  batch_size=bs, shuffle=True,  collate_fn=collate_fn),
            DataLoader(ds_dv,  batch_size=bs, shuffle=False, collate_fn=collate_fn),
            DataLoader(ds_dvr, batch_size=bs, shuffle=False, collate_fn=collate_fn))


def run_encoder_experiment(label, encoder, enc_tokenizer):
    print(f"\n{'='*55}\n  {label}\n{'='*55}")
    tr_loader, dv_loader, dvr_loader = build_enc_loaders(encoder, enc_tokenizer)
    m = BiLSTMClaimClassifier(
        input_dim=768, hidden_dim=256, num_layers=2, num_classes=4, dropout=0.3
    ).to(device)
    m = train_model(m, tr_loader, dvr_loader, device, epochs=15, lr=1e-3)
    ret_acc,  _ = evaluate(m, dvr_loader, device)
    gold_acc, _ = evaluate(m, dv_loader,  device)
    return m, dv_loader, dvr_loader, ret_acc, gold_acc


results = []

# BERT-base (already trained in the previous cell)
bert_ret,  _ = evaluate(model_bilstm, dev_loader_ret, device)
bert_gold, _ = evaluate(model_bilstm, dev_loader,     device)
results.append(('BERT-base', bert_ret, bert_gold,
                model_bilstm, tokenizer, bert, dev_loader_ret))

# ── RoBERTa-base ──────────────────────────────────────────────────────────────
try:
    print("Loading roberta-base...")
    rob_tokenizer = AutoTokenizer.from_pretrained('roberta-base')
    rob_encoder   = AutoModel.from_pretrained('roberta-base').to(device)
    for p in rob_encoder.parameters(): p.requires_grad = False
    rob_encoder.eval()
    model_rob, dev_loader_rob, dev_loader_rob_ret, rob_ret, rob_gold = \
        run_encoder_experiment("RoBERTa-base  (BiLSTM)", rob_encoder, rob_tokenizer)
    results.append(('RoBERTa-base', rob_ret, rob_gold,
                    model_rob, rob_tokenizer, rob_encoder, dev_loader_rob_ret))
except Exception as e:
    print(f"RoBERTa skipped: {e}")

# ── DeBERTa-v3-base ───────────────────────────────────────────────────────────
try:
    print("\nLoading microsoft/deberta-v3-base...")
    deb_tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
    deb_encoder   = AutoModel.from_pretrained('microsoft/deberta-v3-base').to(device)
    for p in deb_encoder.parameters(): p.requires_grad = False
    deb_encoder.eval()
    model_deb, dev_loader_deb, dev_loader_deb_ret, deb_ret, deb_gold = \
        run_encoder_experiment("DeBERTa-v3-base  (BiLSTM)", deb_encoder, deb_tokenizer)
    results.append(('DeBERTa-v3-base', deb_ret, deb_gold,
                    model_deb, deb_tokenizer, deb_encoder, dev_loader_deb_ret))
except Exception as e:
    print(f"DeBERTa skipped: {e}")

print("\n" + "=" * 65)
print(f"{'Encoder':<22} {'Retrieved Dev':>15} {'Gold Dev':>10}")
print("-" * 65)
for name, ret, gold, *_ in results:
    print(f"{name:<22} {ret:>15.4f} {gold:>10.4f}")
print("=" * 65)

# Select the best encoder by retrieved-evidence dev accuracy
best_name, best_ret, best_gold, \
    best_encoder_model, best_encoder_tokenizer, best_encoder, best_dev_loader = \
    max(results, key=lambda r: r[1])

print(f"\nBest encoder: {best_name}  (retrieved dev acc = {best_ret:.4f})")


Loading roberta-base...


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  DeBERTa-v3-base  (BiLSTM)
  -> train (retrieved)...


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

  -> dev (gold)...
  -> dev (retrieved)...
Epoch  1 | Loss 1.4505 | Train Acc 0.2606 | Dev Acc 0.2727
Epoch  2 | Loss 1.3597 | Train Acc 0.3265 | Dev Acc 0.3182
Epoch  3 | Loss 1.3387 | Train Acc 0.3306 | Dev Acc 0.2792
Epoch  4 | Loss 1.3269 | Train Acc 0.3518 | Dev Acc 0.1948
Epoch  5 | Loss 1.2916 | Train Acc 0.3738 | Dev Acc 0.2857
Epoch  6 | Loss 1.2534 | Train Acc 0.4015 | Dev Acc 0.3247
Epoch  7 | Loss 1.2351 | Train Acc 0.4520 | Dev Acc 0.3182
Epoch  8 | Loss 1.1602 | Train Acc 0.4397 | Dev Acc 0.2662
Epoch  9 | Loss 1.1141 | Train Acc 0.4845 | Dev Acc 0.2662
Epoch 10 | Loss 1.0726 | Train Acc 0.4878 | Dev Acc 0.2403
Epoch 11 | Loss 1.0063 | Train Acc 0.5301 | Dev Acc 0.2727
Epoch 12 | Loss 0.9551 | Train Acc 0.5228 | Dev Acc 0.3312
Epoch 13 | Loss 0.9080 | Train Acc 0.5708 | Dev Acc 0.2857
Epoch 14 | Loss 0.8711 | Train Acc 0.5765 | Dev Acc 0.3182
Epoch 15 | Loss 0.7904 | Train Acc 0.6238 | Dev Acc 0.3312

Best dev accuracy: 0.3312

Encoder                  Retrieved Dev   Gol

# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [21]:
# Test predictions - generated the final predictions using best encoder + classifier
# Used pre-computed retrieved evidence from Cell 13
# Output format: {claim_id: {claim_label, evidences}} required by eval.py

_, (preds, labels) = evaluate(best_encoder_model, best_dev_loader, device)
print("Dev set report (retrieved evidence):")
print(classification_report(labels, preds, target_names=[IDX2LABEL[i] for i in range(4)],
                             zero_division=0))

# Use pre-computed test retrieved evidence from Cell 13
test_dataset = ClaimEvidenceDataset(
    test_claims_ret, evidence_corpus, LABEL2IDX,
    best_encoder, best_encoder_tokenizer, device, has_labels=False
)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
_, (test_preds, _) = evaluate(best_encoder_model, test_loader, device)

# Build output with both claim_label AND evidences (required by eval.py)
test_output = {}
for (cid, item), pred in zip(test_claims_ret.items(), test_preds):
    test_output[cid] = {
        'claim_label': IDX2LABEL[pred],
        'evidences': item['evidences']
    }

with open('/content/data/test-claims-predictions.json', 'w') as f:
    json.dump(test_output, f, indent=2)

pred_dist = Counter(v['claim_label'] for v in test_output.values())
print(f"\nPrediction distribution: {dict(pred_dist)}")
print("Saved to /content/data/test-claims-predictions.json")

Dev set report (retrieved evidence):
                 precision    recall  f1-score   support

       SUPPORTS       0.65      0.63      0.64        68
        REFUTES       0.39      0.59      0.47        27
NOT_ENOUGH_INFO       0.49      0.56      0.52        41
       DISPUTED       0.00      0.00      0.00        18

       accuracy                           0.53       154
      macro avg       0.38      0.45      0.41       154
   weighted avg       0.49      0.53      0.51       154


Prediction distribution: {'NOT_ENOUGH_INFO': 44, 'REFUTES': 54, 'SUPPORTS': 54, 'DISPUTED': 1}
Saved to /content/data/test-claims-predictions.json


In [22]:
# Generated dev predictions
# Saved predictions in the required format: {claim_id: {claim_label, evidences}}

dev_output = {}
for (cid, item), pred in zip(dev_claims_ret.items(), preds):
    dev_output[cid] = {
        'claim_label': IDX2LABEL[pred],
        'evidences': item['evidences']
    }

with open('/content/data/dev-claims-predictions.json', 'w') as f:
    json.dump(dev_output, f, indent=2)

print("Dev predictions saved!")

Dev predictions saved!


In [23]:
# Run Official Evaluation Script
# Reports evidence retrieval F-score, claim classification accuracy, and harmonic Mean

!python /content/eval.py \
    --predictions /content/data/dev-claims-predictions.json \
    --groundtruth /content/data/dev-claims.json

python3: can't open file '/content/eval.py': [Errno 2] No such file or directory


## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*

The OOP classes used in this project are defined inline in Section 2:
- ClaimEvidenceDataset (Section 2.8)
- BiLSTMClaimClassifier (Section 2.9)
- UniLSTMClaimClassifier (Section 2.9)